# UDISE+ Mobile Variant — Central Login Test

**Project owner:** Prashant · UMV Tetahali  
**Purpose:** Test whether the Colab runtime can establish the SDMS session starting from the central UDISE login instead of manually pasting SDMS cookies.  
**Mode:** Read-only authentication/session diagnostic. No student update/submit requests are included.  
**Build:** Mobile Variant Login Test · 23 September 2026  
**Open in Colab:** [▶ Launch Mobile Variant Login Test](https://colab.research.google.com/github/zzpsah/udise-automation-suite/blob/main/UDISE_Mobile_Variant_Login_Test_2026-09-23.ipynb)

> This notebook does **not** bypass CAPTCHA/OTP. The user completes any challenge manually. Username/password/session values stay in the current Colab runtime and are never written into the notebook or printed.

### Test target

`Central UDISE login → SDMS / Student Module → current academic session → SDMS session → read-only session check`

Because the central-auth handoff may change, this tester discovers the served form first rather than hard-coding a password POST contract.


In [ ]:
#@title 1️⃣ Setup mobile login test { display-mode: "form" }
!pip install -q requests beautifulsoup4

import re, json, time
from urllib.parse import urljoin, urlparse
from getpass import getpass

import requests
from bs4 import BeautifulSoup
from IPython.display import display, Image, HTML

AUTH_LOGIN_URL = "https://auth.udiseplus.gov.in/login" #@param {type:"string"}
SDMS_BASE_URL = "https://sdms.udiseplus.gov.in" #@param {type:"string"}

auth_session = requests.Session()
auth_session.headers.update({
    "User-Agent": "Mozilla/5.0 (Linux; Android 10; Mobile) AppleWebKit/537.36 Chrome/120 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})

print("✅ Test runtime ready")
print("No credentials or cookies will be printed or saved.")


In [ ]:
#@title 2️⃣ Discover central login form + CAPTCHA { display-mode: "form" }

r = auth_session.get(AUTH_LOGIN_URL, timeout=(15, 45), allow_redirects=True)
print("HTTP:", r.status_code)
print("Final page:", r.url)
if r.status_code != 200:
    raise RuntimeError(f"Central login page returned HTTP {r.status_code}")

soup = BeautifulSoup(r.text, "html.parser")
forms = soup.find_all("form")
if not forms:
    raise RuntimeError("No HTML form was found. The auth page may require JavaScript; stop here and inspect the current portal flow before implementing login.")

def form_score(form):
    txt = (form.get_text(" ", strip=True) + " " + str(form)).lower()
    return sum(k in txt for k in ("password", "captcha", "username", "login", "sign in"))

login_form = max(forms, key=form_score)
FORM_ACTION = urljoin(r.url, login_form.get("action") or r.url)
FORM_METHOD = (login_form.get("method") or "post").lower()

inputs = []
for i in login_form.find_all("input"):
    inputs.append({
        "name": i.get("name") or "",
        "type": (i.get("type") or "text").lower(),
        "id": i.get("id") or "",
        "value_present": bool(i.get("value")),
    })

print("Form method:", FORM_METHOD.upper())
print("Form action:", FORM_ACTION)
print("Input metadata (values hidden):")
for item in inputs:
    print(" -", item)

# Locate a likely CAPTCHA image without attempting to solve it.
captcha_img = None
for img in soup.find_all("img"):
    probe = " ".join(str(img.get(k, "")) for k in ("src", "id", "class", "alt", "title")).lower()
    if "captcha" in probe:
        captcha_img = img
        break

if captcha_img and captcha_img.get("src"):
    CAPTCHA_IMAGE_URL = urljoin(r.url, captcha_img.get("src"))
    cr = auth_session.get(CAPTCHA_IMAGE_URL, timeout=(15, 30))
    if cr.ok and cr.content:
        print("\nCAPTCHA image from the current auth session:")
        display(Image(data=cr.content))
    else:
        print("CAPTCHA image was detected but could not be fetched in the runtime.")
else:
    CAPTCHA_IMAGE_URL = None
    print("\nNo CAPTCHA image URL was identified automatically. If the challenge is JS-rendered, use the browser login/fallback path.")


In [ ]:
#@title 3️⃣ Attempt central login (manual CAPTCHA; credentials hidden) { display-mode: "form" }

if "login_form" not in globals():
    raise RuntimeError("Run the form discovery cell first.")

def pick_input_name(kind):
    candidates = []
    for i in login_form.find_all("input"):
        name = i.get("name") or ""
        iid = i.get("id") or ""
        typ = (i.get("type") or "text").lower()
        probe = (name + " " + iid).lower()
        if kind == "password" and (typ == "password" or "password" in probe or "passwd" in probe):
            candidates.append(name)
        elif kind == "username" and typ not in ("hidden", "password", "submit", "button") and any(k in probe for k in ("user", "login", "userid", "username")):
            candidates.append(name)
        elif kind == "captcha" and "captcha" in probe:
            candidates.append(name)
    return next((x for x in candidates if x), None)

USERNAME_FIELD = pick_input_name("username")
PASSWORD_FIELD = pick_input_name("password")
CAPTCHA_FIELD = pick_input_name("captcha")

print("Detected fields:")
print(" Username:", USERNAME_FIELD or "NOT DETECTED")
print(" Password:", PASSWORD_FIELD or "NOT DETECTED")
print(" CAPTCHA :", CAPTCHA_FIELD or "NOT DETECTED")

if not USERNAME_FIELD or not PASSWORD_FIELD:
    raise RuntimeError("Username/password fields were not safely detected. Do not guess the POST. Inspect the current auth form first.")

username = getpass("UDISE username: ").strip()
password = getpass("UDISE password: ")
captcha = input("CAPTCHA shown above (leave blank only if the current page does not require one): ").strip()

# Preserve all hidden/served form fields exactly as delivered.
payload = {}
for i in login_form.find_all("input"):
    name = i.get("name")
    if name:
        payload[name] = i.get("value") or ""

payload[USERNAME_FIELD] = username
payload[PASSWORD_FIELD] = password
if CAPTCHA_FIELD:
    payload[CAPTCHA_FIELD] = captcha

# Do not print payload or credentials.
headers = {
    "Referer": r.url,
    "Origin": f"{urlparse(r.url).scheme}://{urlparse(r.url).netloc}",
}

if FORM_METHOD == "get":
    lr = auth_session.get(FORM_ACTION, params=payload, headers=headers, timeout=(15, 60), allow_redirects=True)
else:
    lr = auth_session.post(FORM_ACTION, data=payload, headers=headers, timeout=(15, 60), allow_redirects=True)

# Clear local credential variables immediately after request construction/use.
password = None
payload.pop(PASSWORD_FIELD, None)

print("Login response HTTP:", lr.status_code)
print("Final URL:", lr.url)
print("Redirect count:", len(lr.history))
print("Cookie names/domains only (values hidden):")
for c in auth_session.cookies:
    print(f" - {c.name} @ {c.domain}")

body_low = lr.text.lower() if isinstance(lr.text, str) else ""
still_login = ("password" in body_low and "captcha" in body_low and "login" in body_low)
if still_login:
    print("⚠️ The response still looks like a login page.")
    print("This may mean invalid credentials/CAPTCHA, required client-side encryption, or a changed authentication flow.")
    print("No SDMS request will be made automatically from an unconfirmed login.")
else:
    print("✅ Central login appears to have moved beyond the login form. Continue to handoff discovery.")

CENTRAL_LOGIN_RESPONSE = lr


In [ ]:
#@title 4️⃣ Discover SDMS / Student Module / session handoff links { display-mode: "form" }

if "CENTRAL_LOGIN_RESPONSE" not in globals():
    raise RuntimeError("Run the central login attempt first.")

soup2 = BeautifulSoup(CENTRAL_LOGIN_RESPONSE.text, "html.parser")
candidates = []
for a in soup2.find_all("a", href=True):
    href = urljoin(CENTRAL_LOGIN_RESPONSE.url, a["href"])
    label = a.get_text(" ", strip=True)
    probe = (label + " " + href).lower()
    if any(k in probe for k in ("sdms", "student", "module", "session", "academic")):
        candidates.append((label[:100], href))

print(f"Found {len(candidates)} possible handoff links:")
for n, (label, href) in enumerate(candidates[:30], 1):
    print(f"{n:02d}. {label or '(no label)'} -> {href}")

print("\nIf the SDMS/module/session choice is rendered by JavaScript and no useful link appears, use your phone browser to complete that choice and paste the resulting URL in the next cell for investigation.")
MODULE_CANDIDATES = candidates


In [ ]:
#@title 5️⃣ Follow selected module/session handoff { display-mode: "form" }

HANDOFF_URL = "" #@param {type:"string"}

if not HANDOFF_URL.strip():
    print("Paste the SDMS / Student Module / current-session handoff URL above if automatic discovery did not expose a safe target.")
    print("No request sent.")
else:
    hr = auth_session.get(HANDOFF_URL.strip(), timeout=(15, 60), allow_redirects=True)
    print("HTTP:", hr.status_code)
    print("Final URL:", hr.url)
    print("Redirect count:", len(hr.history))
    print("Cookie names/domains only (values hidden):")
    for c in auth_session.cookies:
        print(f" - {c.name} @ {c.domain}")
    HANDOFF_RESPONSE = hr


In [ ]:
#@title 6️⃣ Verify SDMS session read-only { display-mode: "form" }

sdms_cookie_names = {c.name for c in auth_session.cookies if "sdms.udiseplus.gov.in" in (c.domain or "")}
print("SDMS-domain cookie names detected:", sorted(sdms_cookie_names) or "None")

# Build an SDMS-compatible XSRF header only if such a cookie actually exists.
xsrf = None
for c in auth_session.cookies:
    if c.name == "XSRF-TOKEN" and "sdms.udiseplus.gov.in" in (c.domain or ""):
        xsrf = c.value
        break

headers = {
    "Accept": "application/json, text/plain, */*",
    "Referer": f"{SDMS_BASE_URL}/g0/",
    "Origin": SDMS_BASE_URL,
    "User-Agent": "Mozilla/5.0 (Linux; Android 10; Mobile)",
}
if xsrf:
    headers["X-XSRF-TOKEN"] = xsrf

sr = auth_session.get(
    f"{SDMS_BASE_URL}/p0/check-session",
    headers=headers,
    timeout=(15, 30),
    allow_redirects=False,
)

print("check-session HTTP:", sr.status_code)
if 300 <= sr.status_code < 400:
    print("Redirect location:", sr.headers.get("Location", "(not supplied)"))
if sr.status_code == 200:
    print("✅ SDMS session endpoint is reachable from the same runtime session.")
    print("Next step after this test is to confirm the response semantics and then integrate this acquisition flow into the main notebook.")
else:
    print("❌ SDMS session was not confirmed by this read-only check.")
    print("Keep the current Cookie Header login as the baseline fallback; use the diagnostic output above to map the missing handoff step.")


## Interpretation

- This notebook is intentionally a **test harness**, not the production login implementation.
- A successful central-login response alone is not enough; the important proof is that the same runtime reaches the SDMS read-only session endpoint after the Student Module/current-session handoff.
- CAPTCHA/OTP remains manual.
- Do not commit notebook outputs after a real login because output metadata may reveal portal/session details.
- If this test is proven, the production v2.7.3 login can later gain `Central Login` plus the current `Cookie Header` fallback.
